In [ ]:
# Imports
from typing import Literal, List, Union, Optional, Annotated, Tuple, Callable
from collections import defaultdict
import time
import re

# Dependencies
import requests
import pandas as pd
import mwparserfromhell
from pydantic import BaseModel, Field

# Library
from classes.pipeline import VanguardClassifier, VanguardStorage, VanguardScrapper, VanguardParser, MediaWikiAPI, VanguardPipeline
from src.utils import classifier

# Definition
JSONType = dict[str]

In [ ]:
# Params
header = {
	"User-Agent": "VanguardScrapper/1.0 (Python; contact: kmarrero1993@gmail.com)"
}
first_param = {
	"action": "parse",
	"page": "List of Cardfight!! Vanguard Booster Sets",
	"format": "json"
}


In [ ]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(),
	VanguardStorage()
)
curl = pipeline.scrapper.api.get(params=first_param, headers=header)
crude_data = pipeline.scrapper.obtain_links(curl)
crude_consults = pipeline.parser.separate_urls(crude_data)
consults = pipeline.parser.make_consults("get", crude_consults)
pipeline.parser.remove_from_list(crude_consults, [
	"Lyrical Monasterio", "The Mask Collection"
])
pipeline.parser.remove_from_list(crude_data, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])
classifier.process_items(crude_data, pipeline.classifier, pipeline.storage)
classifier.sort_storage_list(["LB", "G"], pipeline.classifier, pipeline.storage)


In [ ]:
curl = pipeline.scrapper.api.get(params=consults[crude_consults[4]], headers=header)
content = pipeline.scrapper.obtain_links(curl)
pipeline.parser.clean_trash_from_set(crude_consults, content, 4)
card_consult = pipeline.parser.make_consults("consult", content)
main_table = pipeline.scrapper.api.get(params=card_consult[0], headers=header)

In [ ]:
card_consult.get(0)

In [ ]:
s = main_table["query"]["pages"]["2010193"]["revisions"][0]["slots"]["main"]["*"]

In [ ]:
main_table

In [ ]:
pages = main_table.get("query", {}).get("pages", {})

In [ ]:
page = next(iter(pages.values()))

In [ ]:
wikitex = page.get("revisions", {})[0].get("slots", {}).get("main", {}).get("*", {})

In [ ]:
c = mwparserfromhell.parse(s)

In [ ]:
for i in c.filter_templates():
	print(i)

In [ ]:
main_table

In [ ]:
def	test_scrap():
	code = mwparserfromhell.parse(s)
	return [template for template in code.filter_templates() if "CardList" in template.name]

In [ ]:
x = test_scrap()

In [ ]:
x